# 6. Fourier et analyse harmonique

**Objectif** : relier la théorie de Fourier aux EDP — montrer que les fonctions propres
du laplacien sont exactement les modes de Fourier, et exploiter la FFT pour résoudre
l'équation de Poisson et l'équation de la chaleur spectralement.

## Plan
1. Série de Fourier et phénomène de Gibbs
2. DFT et FFT : complexité et vérification
3. Lien EDP ↔ Fourier : fonctions propres du laplacien
4. Résolution spectrale de l'équation de Poisson
5. Résolution spectrale de l'équation de la chaleur

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── 1. Série de Fourier de la fonction créneau + phénomène de Gibbs ───────────
# f(x) = sign(sin(x)) sur [0, 2π]
# Coefficients de Fourier : b_k = 4/(kπ) pour k impair, 0 sinon

def fourier_creneau(N_terms, x):
    s = np.zeros_like(x)
    for k in range(1, N_terms+1, 2):   # k impair uniquement
        s += (4.0 / (k * np.pi)) * np.sin(k * x)
    return s

x = np.linspace(0, 2*np.pi, 3000)
f_exact = np.sign(np.sin(x))
f_exact[f_exact == 0] = 0

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, f_exact, 'k-', lw=1.5, label='Créneau exacte')
for N in [5, 15, 50]:
    ax.plot(x, fourier_creneau(N, x), label=f'N={N} termes')
ax.set_title('Série de Fourier du créneau — phénomène de Gibbs (~9% de dépassement)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Dépassement de Gibbs
gibbs_overshoot = np.max(fourier_creneau(1001, x)) - 1.0
print(f'Dépassement de Gibbs : {gibbs_overshoot*100:.2f}%  (valeur théorique ≈ 8.9%)')

In [ ]:
# ── 2. DFT vs FFT : vérification de l'équivalence et gain en complexité ───────

def dft_naive(x):
    """DFT O(N²) : X_k = sum_{n=0}^{N-1} x_n exp(-2pi i k n / N)"""
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    W = np.exp(-2j * np.pi * k * n / N)
    return W @ x

import time
for N in [64, 256, 1024]:
    x_test = np.random.randn(N)
    X_dft  = dft_naive(x_test)
    X_fft  = np.fft.fft(x_test)
    err    = np.max(np.abs(X_dft - X_fft))
    print(f'N={N:5d} : ||DFT - FFT||_inf = {err:.2e}')

In [ ]:
# ── 3. Fonctions propres du laplacien sur [0,1] avec Dirichlet ────────────────
#
# -u'' = λ u,  u(0) = u(1) = 0
# Valeurs propres théoriques : λ_k = k²π²,  k = 1, 2, 3, ...
# Fonctions propres         : φ_k(x) = sqrt(2) sin(kπx)
#
# On vérifie numériquement par la matrice de différences finies.

N  = 200
h  = 1.0 / (N+1)
T  = (1/h**2) * (2*np.eye(N) - np.diag(np.ones(N-1),1) - np.diag(np.ones(N-1),-1))
eigvals_num = np.sort(np.linalg.eigvalsh(T))
k_range     = np.arange(1, 11)
eigvals_th  = (k_range * np.pi)**2

print('Valeurs propres du laplacien discret vs théorie :')
print(f'{"k":>4}  {"λ_k théorique":>18}  {"λ_k numérique":>18}  {"erreur rel."}' )
for k, lth, lnum in zip(k_range, eigvals_th, eigvals_num[:10]):
    print(f'{k:>4}  {lth:>18.6f}  {lnum:>18.6f}  {abs(lth-lnum)/lth:.2e}')

# Visualisation des 4 premières fonctions propres
_, evecs = np.linalg.eigh(T)
x_int = np.linspace(h, 1-h, N)
fig, ax = plt.subplots(figsize=(9, 4))
for k in range(4):
    phi_k  = evecs[:, k] * np.sqrt(2)  # normalisation L2
    phi_th = np.sqrt(2) * np.sin((k+1)*np.pi*x_int)
    # Ajust signe
    if np.dot(phi_k, phi_th) < 0:
        phi_k = -phi_k
    ax.plot(x_int, phi_k, label=f'φ_{k+1} (num.)')
ax.set_title('Fonctions propres du laplacien $-u\'\'$ sur $(0,1)$ = modes de Fourier')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Résolution spectrale de l'équation de Poisson ─────────────────────────
#
# -u'' = f  sur (0,1),  u(0)=u(1)=0
# En base de Fourier sinus : û_k = f̂_k / (k²π²)
# Implémentation via DST (Discrete Sine Transform)

from scipy.fft import dst, idst

N     = 256
h     = 1.0 / (N+1)
x_int = np.linspace(h, 1-h, N)
f_vec = np.pi**2 * np.sin(np.pi * x_int)   # f = π²sin(πx), sol. exacte = sin(πx)

# Coefficients de Fourier sinus de f
f_hat = dst(f_vec, type=1) / (2*(N+1))

# Division par les valeurs propres λ_k = (kπ)²
k       = np.arange(1, N+1)
lam_k   = (k * np.pi)**2
u_hat   = f_hat / lam_k

# Retour dans l'espace physique
u_spectral = idst(u_hat * 2*(N+1), type=1) / (2*(N+1))

u_exact = np.sin(np.pi * x_int)
err_sp  = np.max(np.abs(u_spectral - u_exact))
print(f'Poisson spectral — erreur max = {err_sp:.2e}  (attendu ~ machine precision)')

# ── 5. Résolution spectrale de la chaleur ────────────────────────────────────
#
# ∂_t u = ∂_xx u,  u(0,t)=u(1,t)=0,  u(x,0)=u_0(x)
# Solution : û_k(t) = û_k(0) * exp(-k²π²t)

u0    = lambda x: np.sin(np.pi*x) + 0.5*np.sin(3*np.pi*x)
u0_vec = u0(x_int)
u0_hat = dst(u0_vec, type=1) / (2*(N+1))

times = [0.0, 0.02, 0.1, 0.3]
plt.figure(figsize=(9, 4))
for t in times:
    u_hat_t  = u0_hat * np.exp(-lam_k * t)
    u_t      = idst(u_hat_t * 2*(N+1), type=1) / (2*(N+1))
    plt.plot(x_int, u_t, label=f't={t}')
plt.title('Équation de la chaleur — méthode spectrale (DST)')
plt.xlabel('x'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Mode k=3 se dissipe ~9× plus vite que k=1 (ratio λ_3/λ_1 = 9) : visualisé ✓')

## Bilan : le cercle vertueux Fourier ↔ EDP ↔ espaces de Sobolev

$$\text{Fourier} \longleftrightarrow \text{EDP} \longleftrightarrow \text{Sobolev}$$

- Les **fonctions propres du laplacien** avec conditions de Dirichlet sont les sinus $\phi_k(x) = \sqrt{2}\sin(k\pi x)$.
- L'espace de Sobolev $H^s_0(0,1)$ admet la caractérisation spectrale :
  $$H^s_0(0,1) = \left\{ u = \sum_k \hat{u}_k \phi_k \;:\; \sum_k k^{2s} |\hat{u}_k|^2 < +\infty \right\}.$$
- La **régularité elliptique** se lit dans les coefficients de Fourier : si $f \in L^2$ alors $u \in H^2$
  car $\hat{u}_k = \hat{f}_k / (k^2\pi^2)$ et $\sum k^4 |\hat{u}_k|^2 = \sum |\hat{f}_k|^2 / \pi^4 < +\infty$.
- La **méthode spectrale** est exacte en espace (erreur machine) car elle exploite cette structure propre.